In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Metric libraries
from bert_score import score as bertscore
from rouge_score import rouge_scorer
import sacrebleu

# Paths
SPLIT_DIR = Path("../../processed")
RESULTS_DIR = Path("../../models/06_reframing/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# BERTScore config — fixed for reproducibility
BERTSCORE_MODEL = "roberta-large"
BERTSCORE_LANG = "en"

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


**Load experiment outputs and rebuild the multi-reference lookup**

The output CSV contains only one reference per row (the primary one), but for multi-reference scoring we need the full list of reframes per thought_id from the original test split. Rebuild that lookup here.

In [2]:
df_out = pd.read_csv(RESULTS_DIR / "experiment_outputs.csv")
test_df_raw = pd.read_csv(SPLIT_DIR / "cognitive_reframing_test.csv")

# Multi-reference lookup: thought_id -> list of all reference reframes for that thought
references_by_thought = (
    test_df_raw
    .groupby("thought_id")["reframe"]
    .apply(list)
    .to_dict()
)

# Add a `references` column to df_out for convenience
df_out["references"] = df_out["thought_id"].map(references_by_thought)

print(f"Experiment outputs: {len(df_out)} rows")
print(f"Unique test thoughts: {df_out['thought_id'].nunique()}")
print(f"Reference counts: 1 ref → {sum(1 for r in references_by_thought.values() if len(r)==1)} thoughts, "
      f"2 refs → {sum(1 for r in references_by_thought.values() if len(r)==2)}, "
      f"3 refs → {sum(1 for r in references_by_thought.values() if len(r)==3)}")

Experiment outputs: 228 rows
Unique test thoughts: 57
Reference counts: 1 ref → 1 thoughts, 2 refs → 56, 3 refs → 0


**BERTScore F1 (multi-reference)**

bert-score natively supports multiple references per candidate: pass refs as a list-of-lists, and it returns the max F1 across references for each candidate.

In [3]:
# Prepare candidates and references in matching order
candidates = df_out["generated_reframe"].tolist()
references_lists = df_out["references"].tolist()

print(f"Computing BERTScore F1 on {len(candidates)} candidates...")
print(f"  Model: {BERTSCORE_MODEL}, lang: {BERTSCORE_LANG}")
print(f"  Multi-reference scoring (max F1 across available refs)")

# bert-score handles multi-reference when refs is List[List[str]]
P, R, F1 = bertscore(
    candidates,
    references_lists,
    model_type=BERTSCORE_MODEL,
    lang=BERTSCORE_LANG,
    verbose=True,
    rescale_with_baseline=True,  # important: makes scores interpretable
)

df_out["bertscore_f1"] = F1.numpy()
print(f"\nBERTScore F1 — mean: {F1.mean():.4f}, std: {F1.std():.4f}")

Computing BERTScore F1 on 228 candidates...
  Model: roberta-large, lang: en
  Multi-reference scoring (max F1 across available refs)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/8 [00:00<?, ?it/s]

done in 8.29 seconds, 54.51 sentences/sec

BERTScore F1 — mean: 0.2647, std: 0.0906


**ROUGE-L (multi-reference)**

rouge-score doesn't have built-in multi-reference support, so we compute against each reference and take the max.

In [4]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l_multi_ref(candidate: str, references: list[str]) -> float:
    """Max ROUGE-L F1 across all references."""
    return max(
        scorer.score(ref, candidate)["rougeL"].fmeasure
        for ref in references
    )

print("Computing ROUGE-L (multi-reference, max over refs)...")
df_out["rouge_l"] = [
    rouge_l_multi_ref(cand, refs)
    for cand, refs in zip(candidates, references_lists)
]
print(f"ROUGE-L — mean: {df_out['rouge_l'].mean():.4f}, std: {df_out['rouge_l'].std():.4f}")

Computing ROUGE-L (multi-reference, max over refs)...
ROUGE-L — mean: 0.1783, std: 0.0672


**BLEU-4 (multi-reference)**

sacrebleu supports multi-reference BLEU at the sentence level via sentence_bleu. Multi-reference BLEU at the corpus level (the standard reporting form) requires padding all reference lists to the same length, but sentence-level is fine for per-row scores.

In [5]:
def bleu4_multi_ref(candidate: str, references: list[str]) -> float:
    """Sentence-level BLEU-4 with multi-reference."""
    # sacrebleu.sentence_bleu expects refs as a list of strings (the references)
    result = sacrebleu.sentence_bleu(candidate, references)
    return result.score / 100  # sacrebleu returns 0-100; normalise to 0-1

print("Computing BLEU-4 (multi-reference)...")
df_out["bleu_4"] = [
    bleu4_multi_ref(cand, refs)
    for cand, refs in zip(candidates, references_lists)
]
print(f"BLEU-4 — mean: {df_out['bleu_4'].mean():.4f}, std: {df_out['bleu_4'].std():.4f}")

Computing BLEU-4 (multi-reference)...
BLEU-4 — mean: 0.0393, std: 0.0339


In [6]:
# Drop the `references` column (list-typed, doesn't serialise cleanly)
df_scored = df_out.drop(columns=["references"])
scored_path = RESULTS_DIR / "experiment_outputs_scored.csv"
df_scored.to_csv(scored_path, index=False)
print(f"Saved: {scored_path}")

Saved: ../../models/06_reframing/results/experiment_outputs_scored.csv


In [8]:
def bootstrap_ci(values: np.ndarray, n_boot: int = 10_000, ci: float = 0.95, seed: int = 42):
    """Percentile bootstrap CI for the mean."""
    rng = np.random.default_rng(seed)
    boots = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    lo = np.percentile(boots, (1 - ci) / 2 * 100)
    hi = np.percentile(boots, (1 + ci) / 2 * 100)
    return lo, hi

results = []
for exp_id in ["E1", "E2", "E3", "E4"]:
    sub = df_out[df_out["experiment_id"] == exp_id]
    row = {"experiment": exp_id, "n": len(sub)}
    for metric in ["bertscore_f1", "rouge_l", "bleu_4"]:
        vals = sub[metric].values
        mean = vals.mean()
        lo, hi = bootstrap_ci(vals)
        row[f"{metric}_mean"] = round(mean, 4)
        row[f"{metric}_ci"] = f"[{lo:.4f}, {hi:.4f}]"
    results.append(row)

results_df = pd.DataFrame(results)
print("\n=== Main results table ===\n")
print(results_df.to_string(index=False))

results_df.to_csv(RESULTS_DIR / "main_results.csv", index=False)


=== Main results table ===

experiment  n  bertscore_f1_mean  bertscore_f1_ci  rouge_l_mean       rouge_l_ci  bleu_4_mean        bleu_4_ci
        E1 57             0.2551 [0.2317, 0.2777]        0.1678 [0.1516, 0.1842]       0.0347 [0.0288, 0.0419]
        E2 57             0.2539 [0.2326, 0.2747]        0.1770 [0.1609, 0.1935]       0.0398 [0.0318, 0.0490]
        E3 57             0.2632 [0.2379, 0.2890]        0.1829 [0.1643, 0.2033]       0.0402 [0.0304, 0.0521]
        E4 57             0.2867 [0.2636, 0.3095]        0.1855 [0.1683, 0.2030]       0.0425 [0.0346, 0.0516]


In [9]:
def paired_difference_with_ci(df, metric, exp_a, exp_b, n_boot=10_000, seed=42):
    """
    Paired difference (exp_b - exp_a) per thought_id, with bootstrap CI.
    Positive => exp_b is better.
    """
    a = df[df["experiment_id"] == exp_a].set_index("thought_id")[metric]
    b = df[df["experiment_id"] == exp_b].set_index("thought_id")[metric]
    diffs = (b - a).values  # paired, same thought_id under both conditions

    rng = np.random.default_rng(seed)
    boots = rng.choice(diffs, size=(n_boot, len(diffs)), replace=True).mean(axis=1)
    lo = np.percentile(boots, 2.5)
    hi = np.percentile(boots, 97.5)
    return diffs.mean(), lo, hi

print("\n=== Marginal differences (paired) ===\n")
for metric in ["bertscore_f1", "rouge_l", "bleu_4"]:
    print(f"\n{metric}:")
    for a, b in [("E1", "E2"), ("E2", "E3"), ("E3", "E4")]:
        mean, lo, hi = paired_difference_with_ci(df_out, metric, a, b)
        sig = "***" if (lo > 0 or hi < 0) else "n.s."
        print(f"  {b} - {a}: {mean:+.4f}  CI=[{lo:+.4f}, {hi:+.4f}]  {sig}")


=== Marginal differences (paired) ===


bertscore_f1:
  E2 - E1: -0.0012  CI=[-0.0169, +0.0147]  n.s.
  E3 - E2: +0.0092  CI=[-0.0050, +0.0240]  n.s.
  E4 - E3: +0.0235  CI=[+0.0017, +0.0456]  ***

rouge_l:
  E2 - E1: +0.0092  CI=[-0.0078, +0.0266]  n.s.
  E3 - E2: +0.0059  CI=[-0.0127, +0.0251]  n.s.
  E4 - E3: +0.0026  CI=[-0.0164, +0.0212]  n.s.

bleu_4:
  E2 - E1: +0.0051  CI=[-0.0047, +0.0149]  n.s.
  E3 - E2: +0.0004  CI=[-0.0102, +0.0117]  n.s.
  E4 - E3: +0.0023  CI=[-0.0101, +0.0138]  n.s.


Per-category BERTScore breakdown

In [9]:
per_cat = (
    df_out
    .groupby(["category", "experiment_id"])["bertscore_f1"]
    .mean()
    .unstack("experiment_id")
    .round(4)
    [["E1", "E2", "E3", "E4"]]  # column order
)
per_cat["n_thoughts"] = df_out[df_out["experiment_id"] == "E1"].groupby("category").size()
per_cat = per_cat.sort_values("n_thoughts", ascending=False)

print("\n=== Per-category BERTScore F1 ===\n")
print(per_cat.to_string())

per_cat.to_csv(RESULTS_DIR / "per_category_bertscore.csv")


=== Per-category BERTScore F1 ===

experiment_id                    E1      E2      E3      E4  n_thoughts
category                                                               
all-or-nothing thinking      0.2751  0.2828  0.2869  0.2684           8
negative feeling or emotion  0.3149  0.3063  0.2856  0.3104           8
labeling                     0.2424  0.2207  0.2324  0.2919           7
mind reading                 0.2238  0.2071  0.2355  0.2743           7
catastrophizing              0.2230  0.2242  0.1723  0.2130           5
fortune telling              0.3808  0.2994  0.3385  0.3557           5
emotional reasoning          0.2625  0.3119  0.3109  0.2923           4
disqualifying the positive   0.2351  0.2579  0.1992  0.2533           3
overgeneralizing             0.2533  0.2216  0.2981  0.2904           3
personalizing                0.2883  0.3254  0.2352  0.2768           3
blaming                      0.2224  0.2487  0.2129  0.1938           2
comparing and despairing    